# Presentation preprocessing and recording EDA

Rebuild the analytical plots from authorized recordings. Page numbers refer to the original 48-page presentation. The charts use the checked-in readers; the exact original artist scripts and a few example selections are not known, so visual reconstructions are marked below. Run the [brain activity notebook](brain_activity_eda.ipynb) for the original Mitt odor/channel and correctness panels (pp. 44–45, 47).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / 'src').is_dir() and (p / 'poster').is_dir())
sys.path.insert(0, str(root / 'src'))
from data_processing import preprocess
from data_processing import presentation_signal as signal
from data_processing.vendor.io import metadata, rows
from data_processing.storage import lock

raw = root / 'data/raw_data'
pairs = preprocess.discover(raw)
mitt = signal.SESSIONS['Mitt']
superchris = signal.SESSIONS['Superchris']
cache = root / 'results/presentation-eda'

## Alignment, ownership and rejection (pp. 8–12)

The alignment panel uses a real 200 ms odor window (illustration reselected). Odor uses correct InSeq trials; position also includes correct OutSeq trials. The split diagram shows the actual chronological raw-sample fractions and counts real windows that cross either split boundary.

In [ ]:
trial, traces, markers = signal.alignment_example(pairs[mitt])
x = np.arange(-250, 450)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, layout='constrained')
for i, (name, trace) in enumerate(traces.items()):
    axes[0].plot(x, (trace - np.mean(trace)) / np.std(trace) + 8*i, label=name)
for i, (name, marker) in enumerate(markers.items()):
    axes[1].step(x, marker + 3*i, where='post', label=name)
for ax in axes:
    ax.axvspan(0, 200, alpha=.12, color='tab:orange')
    ax.axvline(0, color='black', linestyle=':')
    ax.legend(ncols=4)
axes[0].set(title=f'Mitt, trial {trial["trial_id"]}: aligned LFP and behavior', ylabel='LFP (offset for display)')
axes[1].set(xlabel='Time from odor onset (ms)', ylabel='Behavior (offset for display)');

examples, center, limit, assigned = signal.trial_quality(pairs[superchris])
fig, axes = plt.subplots(1, 2, sharey=True, figsize=(9, 3), layout='constrained')
for ax, (trial_id, trace) in zip(axes, examples.items(), strict=True):
    ax.axhspan(center - limit, center + limit, color='0.9', label='Allowed training-derived range')
    reject = bool(np.any(np.abs(trace - center) > limit))
    ax.plot(trace, color='tab:red' if reject else 'tab:blue')
    ax.set(title=f'Superchris T1, trial {trial_id}: ' + ('amplitude artifact' if reject else 'within amplitude bounds'),
           xlabel='Time from odor onset (ms)')
axes[0].set_ylabel('LFP (stored units)')
axes[0].legend(fontsize=8);

n_samples = metadata(pairs[superchris][0])['shape'][1]
fig, ax = plt.subplots(figsize=(10, 2.5), layout='constrained')
for a, b, name, color in [(0, .7, 'train', 'tab:blue'), (.7, .85, 'validation', 'tab:orange'), (.85, 1, 'test', 'tab:green')]:
    ax.axvspan(a, b, color=color, alpha=.15, label=name)
    starts = assigned.loc[assigned.split.eq(name), 'start'] / n_samples
    ax.scatter(starts, np.zeros(len(starts)), marker='|', color=color)
crossed = assigned.reasons.map(lambda reasons: 'split_boundary' in reasons)
ax.scatter(assigned.loc[crossed, 'start'] / n_samples, np.zeros(crossed.sum()), marker='x', color='red')
ax.set(xlabel='Fraction of recorded samples', yticks=[], title=f'Superchris: {crossed.sum()} windows cross split boundaries')
ax.legend(ncols=3);

## Training-only normalization and Morlet features (pp. 13, 15)

The normalization panel uses the slide’s Mitt trial 0, with raw T10/T19 voltage and a z-score fitted on unique samples from matched training trials; the exact scaler originally used to draw the slide is unknown. The wavelet panel uses the actual 12 Morlet coefficients per channel for that trial. The model stores phase as sine and cosine, so its combined representation has **792** inputs per sample for 22 channels, even though the slide shows amplitude and angle as **528** conceptual quantities.

In [ ]:
with lock(cache / '.preparation.lock'):
    preprocess.prepare(mitt, pairs[mitt], cache)
base = cache / 'preprocessing' / mitt / 'odor'
import pandas as pd
frame = pd.read_csv(base / 'matched_trials.csv')
train_samples = np.unique((frame.loc[frame.split.eq('train'), 'onset'].to_numpy()[:, None] + np.arange(500)).ravel())
from json import loads
channels = [k.split('_', 1)[0] for k in loads((base / 'qa.json').read_text())['channel_keys']]
i_trial = frame.index[frame.trial_id.eq(0)].item()
start = frame.loc[i_trial, 'onset']
assert start == 87791 and frame.loc[i_trial, 'split'] == 'train'
fig, axes = plt.subplots(2, 2, sharex=True, sharey='col', figsize=(10, 5), layout='constrained')
voltage = signal.selected_channels(pairs[mitt][0], ('T10', 'T19'))
for i, name in enumerate(('T10', 'T19')):
    row = voltage[name]
    trace, fit = row[start:start+500], row[train_samples]
    axes[i, 0].plot(trace)
    axes[i, 1].plot((trace - fit.mean()) / fit.std())
    axes[i, 0].set_ylabel(name)
axes[0, 0].set_title('Raw voltage (stored units)')
axes[0, 1].set_title('After training-only z-score')
fig.supxlabel('Time from odor onset (ms)');

c = channels.index('T1')
amplitude = np.load(base / 'amp.npy', mmap_mode='r', allow_pickle=False)[i_trial, :, c*12:(c+1)*12]
phase = np.load(base / 'phase.npy', mmap_mode='r', allow_pickle=False)[i_trial, :, c*24:(c+1)*24]
phase = np.arctan2(phase[:, ::2], phase[:, 1::2])
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, layout='constrained')
for ax, matrix, name, cmap in zip(axes, (amplitude, phase), ('Log amplitude', 'Phase (radians)'), ('viridis', 'twilight'), strict=True):
    image = ax.imshow(matrix.T, aspect='auto', origin='lower', cmap=cmap)
    ax.set(yticks=range(12), yticklabels=np.round(preprocess.FREQUENCIES, 1), ylabel='Frequency (Hz)', title=name)
    fig.colorbar(image, ax=ax)
axes[-1].set_xlabel(f'Mitt T1, trial {frame.iloc[0].trial_id} · time after odor onset (ms)');

## 60 Hz spectral spike (p. 14)

Recomputed raw PSD: complete nonoverlapping 10-second blocks within clock-contiguous segments; Welch PSD within each block (4-second Hann, 50% overlap); median across blocks per channel, then channel median and interquartile range. The second panel compares the same Superchris T1 training interval before and after a 60 Hz notch. The slide shows an electrical line in the LFP spectrum, not spike-unit activity.

In [ ]:
summaries = signal.recording_stats(pairs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
for name, stat in summaries.items():
    hz = stat['hz']
    keep = (hz >= .5) & (hz <= 250)
    lo, mid, hi = 10*np.log10(np.quantile(stat['psd'][:, keep], (.25, .5, .75), axis=0))
    line, = axes[0].plot(hz[keep], mid, label=name)
    axes[0].fill_between(hz[keep], lo, hi, color=line.get_color(), alpha=.1)
axes[0].set(xscale='log', title='Raw PSD across five sessions')
hz, before, after = signal.notch_spectrum(pairs[superchris])
keep = (hz >= 50) & (hz <= 70)
axes[1].plot(hz[keep], 10*np.log10(before[keep]), label='Before notch')
axes[1].plot(hz[keep], 10*np.log10(after[keep]), label='After notch')
axes[1].set_title('Paired Superchris T1 PSD')
for ax in axes:
    ax.axvline(60, color='0.4', linestyle=':')
    ax.set(xlabel='Frequency (Hz)', ylabel='PSD (dB re stored unit²/Hz)')
    ax.legend();

## Signal quality and heatmaps (pp. 39–42)

The amplitude heatmap compares each channel with its own median 10-second window SD, not between-channel voltage. Gray entries in cross-session maps indicate absent channels. The labels do not establish anatomical correspondence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), layout='constrained')
barat = signal.selected_channels(pairs[signal.SESSIONS['Barat']][0], ('T2', 'T8', 'T22'))
clip = 1_067_139
for name, trace in barat.items():
    axes[0].plot(np.arange(2000)/1000, trace[clip:clip+2000], label=name)
axes[0].set(title='Barat: weak channels alongside T8', xlabel='Seconds into clip', ylabel='LFP')
length, key, onset = summaries['Buchanan']['plateau']
stat = summaries['Buchanan']
a, b = max(0, onset-1000), min(len(stat['time']), onset+length+1000)
for name, trace in signal.selected_channels(pairs[signal.SESSIONS['Buchanan']][0], {key, 'T6', 'T18'}).items():
    axes[1].plot(stat['time'][a:b], trace[a:b], label=name)
axes[1].axvspan(stat['time'][onset], stat['time'][onset+length-1], alpha=.2, color='tab:orange')
axes[1].set(title=f'Buchanan: longest exact plateau ({length} ms)', xlabel='Recording clock (s)')
for ax in axes:
    ax.legend();

fig, axes = plt.subplots(len(summaries), 1, figsize=(11, 6), layout='constrained')
for ax, (name, stat) in zip(axes, summaries.items(), strict=True):
    image = ax.imshow(stat['local_change'], aspect='auto', vmin=-2, vmax=2, cmap='RdBu_r')
    for cut in stat['cuts']:
        ax.axvline(cut - .5, color='0.2', linestyle=':', linewidth=.8)
    ax.set(ylabel=name, yticks=[])
axes[-1].set_xlabel('Recorded 10-second window order (gaps compressed)')
fig.colorbar(image, ax=axes, label='log₂(window SD / channel median window SD)');

fig, axes = plt.subplots(1, 3, figsize=(12, 6), layout='constrained')
for ax, (metric, title, cmap) in zip(axes, [('present', 'Channel availability', 'Blues'), ('relative_sd', 'SD relative to session median (log₂)', 'RdBu_r'), ('line_db', 'Peak near 60 Hz above flanks (dB)', 'viridis')], strict=True):
    panel = np.full((len(signal.CHANNELS), len(summaries)), np.nan)
    for j, stat in enumerate(summaries.values()):
        for i, key in enumerate(stat['keys']):
            panel[signal.CHANNELS.index(key), j] = 1 if metric == 'present' else stat[metric][i]
    ax.set_facecolor('0.88')
    im = ax.imshow(panel, aspect='auto', cmap=cmap, vmin=0 if metric == 'present' else None)
    ax.set(title=title, xticks=range(len(summaries)), yticks=range(len(signal.CHANNELS)), yticklabels=signal.CHANNELS)
    ax.set_xticklabels(summaries, rotation=30, ha='right')
    if metric != 'present':
        fig.colorbar(im, ax=ax, shrink=.8);

## Odor indicator and inter-odor transitions (pp. 43, 46)

Odor 3 is a single-sample delivery marker, not a sustained state. The plot retains every active marker and thins the inactive samples for display. For the lower panel, group departures from correct InSeq odors A–D by departing odor, keep the next poke within 10 seconds in the same segment, and truncate each displayed trace at that poke. This rule reproduces the slide’s group counts but does not require the next odor to be the nominal successor.

In [ ]:
bvr = pairs[mitt][1]
odor3 = next(row for k, row in zip(metadata(bvr)['keys'], rows(bvr), strict=True) if k == 'Odor3')
time = summaries['Mitt']['time']
active = odor3 == 1
background = np.arange(0, len(time), 100)
background = background[~active[background]]
fig, ax = plt.subplots(figsize=(9, 2), layout='constrained')
ax.plot(time[background], odor3[background], '.', markersize=1)
ax.plot(time[active], odor3[active], '.', markersize=2)
ax.set(title='Mitt: raw Odor 3 indicator', xlabel='Recording clock (s)', ylabel='Marker');

averages = signal.transition_averages(pairs[mitt])
fig, axes = plt.subplots(2, 1, figsize=(9, 6), layout='constrained')
for name, (values, count) in sorted(averages.items()):
    axes[0 if name.startswith('Odor') else 1].plot((np.arange(len(values))-500)/1000, values, label=f'{name} (n={count})')
for ax, name in zip(axes, ('Odor · aligned to poke entry', 'Between odors · aligned to withdrawal'), strict=True):
    ax.axvline(0, color='0.4', linestyle=':')
    ax.set(title=name, xlabel='Time from event (s)', ylabel='Mean baseline-corrected LFP')
    ax.legend(ncols=3);